In [5]:
!pip install sqlalchemy psycopg2-binary


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
!pip install holidays

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   -------------- ------------------------- 0.5/1.4 MB 372.4 kB/s eta 0:00:03
   -------------- ------------------------- 0.5/1.4 MB 372.4 kB/s eta 0:00:03
   -------------- ------------------------- 0.5/1.4 MB 372.4 kB/s eta 0:00:03
   -------------- ------------------------- 0.5/1.4 MB 372.4 kB/s eta 0:00:03
   -------------- ------------------------- 0.5/1.4 MB 372.4 kB/s eta 0:00:03
   ---------------------- ----------------- 0.8/1.4 MB 307.7 kB/s eta 0:00:03
   ---------------------- ----------------- 0.8/1.4 MB 307.7 kB/s eta 0:00:03
   ---------------------- ----------------- 0.8/1.4 MB 307.7 kB/s eta 0:00:03
   ---------------------- -----


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import pandas as pd
import requests
import holidays
from sqlalchemy import create_engine, text

# =========================================================
# 1. PostgreSQL-Verbindung
# =========================================================
DB_USER = "postgres"
DB_PASSWORD = "DSI"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "spritdb"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# =========================================================
# 2. Hilfsfunktionen
# =========================================================
def lade_nrw_feiertage(jahre):
    rows = []
    try:
        de_holidays = holidays.country_holidays("DE", subdiv="NW", years=jahre)
        for i, (datum, name) in enumerate(sorted(de_holidays.items()), start=1):
            rows.append({
                "feiertag_id": i,
                "datum": pd.to_datetime(datum).date(),
                "name": str(name),
                "bundesland": "NRW"
            })
    except Exception as e:
        print("Fehler beim Laden der Feiertage:", e)

    return pd.DataFrame(rows, columns=["feiertag_id", "datum", "name", "bundesland"])


def lade_nrw_schulferien(jahre):
    rows = []
    next_id = 1

    for jahr in jahre:
        try:
            url = f"https://ferien-api.de/api/v1/holidays/NW/{jahr}"
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            daten = response.json()

            if isinstance(daten, dict):
                daten = list(daten.values())

            for eintrag in daten:
                start = eintrag.get("start")
                end = eintrag.get("end")

                if start and end:
                    rows.append({
                        "schulferien_id": next_id,
                        "startdatum": pd.to_datetime(start, errors="coerce").date(),
                        "enddatum": pd.to_datetime(end, errors="coerce").date(),
                        "bundesland": "NRW",
                        "name": eintrag.get("name", None)
                    })
                    next_id += 1
        except Exception as e:
            print(f"Warnung: Schulferien für {jahr} konnten nicht geladen werden: {e}")

    return pd.DataFrame(
        rows,
        columns=["schulferien_id", "startdatum", "enddatum", "bundesland", "name"]
    )


def finde_schulferien_id(datum, schulferien_df):
    if schulferien_df.empty:
        return None

    treffer = schulferien_df[
        (schulferien_df["startdatum"] <= datum) &
        (schulferien_df["enddatum"] >= datum)
    ]

    if len(treffer) > 0:
        return int(treffer.iloc[0]["schulferien_id"])

    return None


# =========================================================
# 3. Tabellen anlegen / Schema korrigieren
# =========================================================
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS marken (
            marken_id BIGINT PRIMARY KEY,
            marken_name VARCHAR(100) NOT NULL UNIQUE
        )
    """))

    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS staedte (
            staedte_id BIGINT PRIMARY KEY,
            staedte_name VARCHAR(100) NOT NULL UNIQUE
        )
    """))

    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS kraftstoff (
            kraftstoff_id INT PRIMARY KEY,
            name VARCHAR(20) NOT NULL UNIQUE
        )
    """))

    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS feiertag (
            feiertag_id BIGINT PRIMARY KEY,
            datum DATE NOT NULL,
            name VARCHAR(100),
            bundesland VARCHAR(100)
        )
    """))

    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS schulferien (
            schulferien_id BIGINT PRIMARY KEY,
            startdatum DATE NOT NULL,
            enddatum DATE NOT NULL,
            bundesland VARCHAR(100),
            name VARCHAR(100)
        )
    """))

    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS tankstelle (
            station_uuid VARCHAR(100) PRIMARY KEY,
            marken_id BIGINT,
            city_id BIGINT,
            station_name VARCHAR(150),
            FOREIGN KEY (marken_id) REFERENCES marken(marken_id),
            FOREIGN KEY (city_id) REFERENCES staedte(staedte_id)
        )
    """))

    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS kalender (
            zeit_id BIGINT PRIMARY KEY,
            zeitstempel TIMESTAMP NOT NULL,
            jahr INT NOT NULL,
            monat INT NOT NULL,
            tag INT NOT NULL,
            stunde INT NOT NULL,
            minute INT NOT NULL,
            sekunde INT NOT NULL,
            wochentag INT NOT NULL,
            feiertag_id BIGINT,
            schulferien_id BIGINT,
            FOREIGN KEY (feiertag_id) REFERENCES feiertag(feiertag_id),
            FOREIGN KEY (schulferien_id) REFERENCES schulferien(schulferien_id)
        )
    """))

    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS preise (
            preis_id BIGINT PRIMARY KEY,
            station_uuid VARCHAR(100) NOT NULL,
            zeit_id BIGINT NOT NULL,
            kraftstoff_id INT NOT NULL,
            preis DECIMAL(6,3) NOT NULL,
            FOREIGN KEY (station_uuid) REFERENCES tankstelle(station_uuid),
            FOREIGN KEY (zeit_id) REFERENCES kalender(zeit_id),
            FOREIGN KEY (kraftstoff_id) REFERENCES kraftstoff(kraftstoff_id)
        )
    """))

    # Falls Tabellen schon früher ohne diese Spalten angelegt wurden:
    conn.execute(text("""
        ALTER TABLE feiertag
        ADD COLUMN IF NOT EXISTS name VARCHAR(100)
    """))
    conn.execute(text("""
        ALTER TABLE feiertag
        ADD COLUMN IF NOT EXISTS bundesland VARCHAR(100)
    """))
    conn.execute(text("""
        ALTER TABLE schulferien
        ADD COLUMN IF NOT EXISTS bundesland VARCHAR(100)
    """))
    conn.execute(text("""
        ALTER TABLE schulferien
        ADD COLUMN IF NOT EXISTS name VARCHAR(100)
    """))

print("Tabellen wurden angelegt oder aktualisiert.")

# =========================================================
# 4. CSV laden
# =========================================================
df = pd.read_csv("ARAL_tagespreise.csv")
df = df.drop(columns=["Unnamed: 0"], errors="ignore")
df.columns = df.columns.str.strip()

print("CSV geladen.")
print(df.head())
print("Spalten:", df.columns.tolist())
print("Zeilen/Spalten:", df.shape)

# =========================================================
# 5. Pflichtspalten prüfen
# =========================================================
basis_spalten = ["station_uuid", "diesel", "e5", "e10", "datum", "uhrzeit"]
for col in basis_spalten:
    if col not in df.columns:
        raise ValueError(f"Spalte '{col}' fehlt in der CSV-Datei.")

# =========================================================
# 6. Optionale Spalten ergänzen
# =========================================================
if "marke" not in df.columns:
    df["marke"] = "Aral"

if "stadt" not in df.columns:
    df["stadt"] = "Unbekannt"

if "station_name" not in df.columns:
    df["station_name"] = "Aral Tankstelle"

# =========================================================
# 7. Preisspalten bereinigen
# =========================================================
fuel_cols = ["diesel", "e5", "e10"]

df[fuel_cols] = df[fuel_cols].replace(["null", "NULL", "", "nan", "NaN", 0], pd.NA)

for col in fuel_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[fuel_cols] = df[fuel_cols].ffill().bfill()

# =========================================================
# 8. Datum + Uhrzeit zusammenführen
# =========================================================
df["datetime_str"] = df["datum"].astype(str).str.strip() + " " + df["uhrzeit"].astype(str).str.strip()

df["datetime"] = pd.to_datetime(df["datetime_str"], errors="coerce")

print(df[["datum", "uhrzeit", "datetime_str", "datetime"]].head())
print("Gültige datetime:", df["datetime"].notna().sum())
print("Ungültige datetime:", df["datetime"].isna().sum())

df = df.dropna(subset=["datetime"]).copy()

# =========================================================
# 9. Jahre bestimmen
# =========================================================
jahre = sorted(df["datetime"].dt.year.dropna().unique().tolist())
print("Gefundene Jahre:", jahre)

# =========================================================
# 10. Marken-Tabelle
# =========================================================
marken = (
    df[["marke"]]
    .drop_duplicates()
    .sort_values("marke")
    .reset_index(drop=True)
    .copy()
)
marken["marken_id"] = range(1, len(marken) + 1)
marken = marken.rename(columns={"marke": "marken_name"})
marken = marken[["marken_id", "marken_name"]]

# =========================================================
# 11. Städte-Tabelle
# =========================================================
staedte = (
    df[["stadt"]]
    .drop_duplicates()
    .sort_values("stadt")
    .reset_index(drop=True)
    .copy()
)
staedte["staedte_id"] = range(1, len(staedte) + 1)
staedte = staedte.rename(columns={"stadt": "staedte_name"})
staedte = staedte[["staedte_id", "staedte_name"]]

# =========================================================
# 12. IDs in Haupt-DataFrame mergen
# =========================================================
df = df.merge(marken, left_on="marke", right_on="marken_name", how="left")
df = df.merge(staedte, left_on="stadt", right_on="staedte_name", how="left")

# =========================================================
# 13. Tankstelle-Tabelle
# =========================================================
tankstelle = (
    df[["station_uuid", "marken_id", "staedte_id", "station_name"]]
    .drop_duplicates()
    .copy()
)
tankstelle = tankstelle.rename(columns={"staedte_id": "city_id"})
tankstelle = tankstelle[["station_uuid", "marken_id", "city_id", "station_name"]]

# =========================================================
# 14. Kraftstoff-Tabelle
# =========================================================
kraftstoff = pd.DataFrame({
    "kraftstoff_id": [1, 2, 3],
    "name": ["diesel", "e5", "e10"]
})

# =========================================================
# 15. Feiertage und Schulferien laden
# =========================================================
feiertag = lade_nrw_feiertage(jahre)
schulferien = lade_nrw_schulferien(jahre)

print("Feiertage geladen:", len(feiertag))
print("Schulferien geladen:", len(schulferien))

# =========================================================
# 16. Preisdaten ins Long-Format
# =========================================================
df_long = df.melt(
    id_vars=["station_uuid", "datetime"],
    value_vars=fuel_cols,
    var_name="kraftstoff_typ",
    value_name="preis"
)

df_long["datetime"] = pd.to_datetime(df_long["datetime"], errors="coerce")
df_long = df_long.dropna(subset=["datetime", "preis"]).copy()

kraftstoff_map = {"diesel": 1, "e5": 2, "e10": 3}
df_long["kraftstoff_id"] = df_long["kraftstoff_typ"].map(kraftstoff_map)
df_long = df_long.dropna(subset=["kraftstoff_id"]).copy()
df_long["kraftstoff_id"] = df_long["kraftstoff_id"].astype(int)

print("Long-Format Zeilen:", len(df_long))

# =========================================================
# 17. Kalender-Tabelle
# =========================================================
kalender = pd.DataFrame({
    "zeitstempel": df_long["datetime"].drop_duplicates()
})

kalender["zeitstempel"] = pd.to_datetime(kalender["zeitstempel"], errors="coerce")
kalender = (
    kalender
    .dropna(subset=["zeitstempel"])
    .sort_values("zeitstempel")
    .reset_index(drop=True)
)

kalender["zeit_id"] = range(1, len(kalender) + 1)
kalender["jahr"] = kalender["zeitstempel"].dt.year
kalender["monat"] = kalender["zeitstempel"].dt.month
kalender["tag"] = kalender["zeitstempel"].dt.day
kalender["stunde"] = kalender["zeitstempel"].dt.hour
kalender["minute"] = kalender["zeitstempel"].dt.minute
kalender["sekunde"] = kalender["zeitstempel"].dt.second
kalender["wochentag"] = kalender["zeitstempel"].dt.weekday + 1
kalender["datum"] = kalender["zeitstempel"].dt.date

kalender["feiertag_id"] = None
kalender["schulferien_id"] = None

if not feiertag.empty and {"feiertag_id", "datum"}.issubset(feiertag.columns):
    feiertag["datum"] = pd.to_datetime(feiertag["datum"], errors="coerce").dt.date
    kalender = kalender.merge(
        feiertag[["feiertag_id", "datum"]],
        on="datum",
        how="left",
        suffixes=("", "_neu")
    )
    if "feiertag_id_neu" in kalender.columns:
        kalender["feiertag_id"] = kalender["feiertag_id_neu"]
        kalender = kalender.drop(columns=["feiertag_id_neu"])

if not schulferien.empty and {"schulferien_id", "startdatum", "enddatum"}.issubset(schulferien.columns):
    schulferien["startdatum"] = pd.to_datetime(schulferien["startdatum"], errors="coerce").dt.date
    schulferien["enddatum"] = pd.to_datetime(schulferien["enddatum"], errors="coerce").dt.date
    kalender["schulferien_id"] = kalender["datum"].apply(
        lambda d: finde_schulferien_id(d, schulferien)
    )

kalender = kalender.drop(columns=["datum"])

print("Kalender-Zeilen:", len(kalender))

# =========================================================
# 18. zeit_id in Preistabelle mergen
# =========================================================
df_long = df_long.merge(
    kalender[["zeit_id", "zeitstempel"]],
    left_on="datetime",
    right_on="zeitstempel",
    how="left"
)

preise = df_long[["station_uuid", "zeit_id", "kraftstoff_id", "preis"]].copy()
preise = preise.dropna(subset=["station_uuid", "zeit_id", "kraftstoff_id", "preis"]).reset_index(drop=True)

preise["zeit_id"] = preise["zeit_id"].astype(int)
preise["kraftstoff_id"] = preise["kraftstoff_id"].astype(int)
preise["preis_id"] = range(1, len(preise) + 1)
preise = preise[["preis_id", "station_uuid", "zeit_id", "kraftstoff_id", "preis"]]

print("Preise-Zeilen:", len(preise))

# =========================================================
# 19. Alte Daten löschen
# =========================================================
with engine.begin() as conn:
    conn.execute(text("DELETE FROM preise"))
    conn.execute(text("DELETE FROM kalender"))
    conn.execute(text("DELETE FROM tankstelle"))
    conn.execute(text("DELETE FROM feiertag"))
    conn.execute(text("DELETE FROM schulferien"))
    conn.execute(text("DELETE FROM kraftstoff"))
    conn.execute(text("DELETE FROM marken"))
    conn.execute(text("DELETE FROM staedte"))

print("Alte Daten gelöscht.")

# =========================================================
# 20. Neue Daten schreiben
# =========================================================
with engine.begin() as conn:
    staedte.to_sql("staedte", conn, if_exists="append", index=False)
    marken.to_sql("marken", conn, if_exists="append", index=False)
    kraftstoff.to_sql("kraftstoff", conn, if_exists="append", index=False)

    if not feiertag.empty:
        feiertag.to_sql("feiertag", conn, if_exists="append", index=False)

    if not schulferien.empty:
        schulferien.to_sql("schulferien", conn, if_exists="append", index=False)

    tankstelle.to_sql("tankstelle", conn, if_exists="append", index=False)
    kalender.to_sql("kalender", conn, if_exists="append", index=False)
    preise.to_sql("preise", conn, if_exists="append", index=False)

print("Import erfolgreich abgeschlossen.")

# =========================================================
# 21. Testabfragen
# =========================================================
print("\n--- Feiertage ---")
if not feiertag.empty:
    print(pd.read_sql("SELECT * FROM feiertag LIMIT 10", engine))
else:
    print("Keine Feiertage geladen.")

print("\n--- Schulferien ---")
if not schulferien.empty:
    print(pd.read_sql("SELECT * FROM schulferien LIMIT 10", engine))
else:
    print("Keine Schulferien geladen.")

print("\n--- Kalender ---")
print(pd.read_sql("""
    SELECT zeit_id, zeitstempel, feiertag_id, schulferien_id
    FROM kalender
    ORDER BY zeitstempel
    LIMIT 20
""", engine))

print("\n--- Preise ---")
print(pd.read_sql("""
    SELECT p.preis_id, p.preis, k.zeitstempel, kr.name AS kraftstoff
    FROM preise p
    JOIN kalender k ON p.zeit_id = k.zeit_id
    JOIN kraftstoff kr ON p.kraftstoff_id = kr.kraftstoff_id
    ORDER BY k.zeitstempel
    LIMIT 10
""", engine))

Tabellen wurden angelegt oder aktualisiert.
CSV geladen.
                           station_uuid  diesel     e5    e10       datum  \
0  e1aefc4e-3ca1-4018-8d91-455b69d35d41   1.099  1.349  1.329  2017-06-09   
1  e1aefc4e-3ca1-4018-8d91-455b69d35d41   1.099  1.339  1.319  2017-06-09   
2  e1aefc4e-3ca1-4018-8d91-455b69d35d41   1.089  1.339  1.319  2017-06-09   
3  e1aefc4e-3ca1-4018-8d91-455b69d35d41   1.079  1.319  1.299  2017-06-09   
4  e1aefc4e-3ca1-4018-8d91-455b69d35d41   1.099  1.339  1.319  2017-06-09   

    uhrzeit  
0  09:52:06  
1  10:18:07  
2  11:37:07  
3  14:08:06  
4  15:01:06  
Spalten: ['station_uuid', 'diesel', 'e5', 'e10', 'datum', 'uhrzeit']
Zeilen/Spalten: (78948, 6)
        datum   uhrzeit         datetime_str            datetime
0  2017-06-09  09:52:06  2017-06-09 09:52:06 2017-06-09 09:52:06
1  2017-06-09  10:18:07  2017-06-09 10:18:07 2017-06-09 10:18:07
2  2017-06-09  11:37:07  2017-06-09 11:37:07 2017-06-09 11:37:07
3  2017-06-09  14:08:06  2017-06-09 14:0